<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading_V0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V0 Fixed Grid Backtest

This notebook runs `Grid_trading_V0.py` from the current `main` branch. V0 uses user-defined Initial Capital, Floor, Ceiling, Gap, and Fees. After the run it writes `logs/latest_v0_backtest_log.json` so the latest result can be reviewed directly from GitHub.

## 1. Load current V0 code and mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import base64
import importlib.util

import numpy as np
import pandas as pd
import requests

REPO = 'natdanaiii/Trading'
BRANCH = 'main'
SOURCE_FILE = 'Grid_trading_V0.py'
GITHUB_LOG_PATH = 'logs/latest_v0_backtest_log.json'
LOCAL_SOURCE_PATH = '/content/Grid_trading_V0.py'
LOCAL_LOG_PATH = '/content/latest_v0_backtest_log.json'

source_url = f'https://raw.githubusercontent.com/{REPO}/{BRANCH}/{SOURCE_FILE}'
response = requests.get(source_url, timeout=30)
response.raise_for_status()

with open(LOCAL_SOURCE_PATH, 'w', encoding='utf-8') as f:
    f.write(response.text)

spec = importlib.util.spec_from_file_location('grid_v0', LOCAL_SOURCE_PATH)
v0mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(v0mod)

print(f'Loaded current {SOURCE_FILE} from {REPO}/{BRANCH}')

## 2. Configuration and backtest

In [ ]:
df_1m = v0mod.load_market_data(
    v0mod.SYMBOL,
    v0mod.TIMEFRAME,
    v0mod.DATA_DIR,
)

grid = v0mod.build_fixed_grid_table(
    capital=v0mod.INITIAL_CAPITAL,
    floor=v0mod.GRID_FLOOR,
    ceiling=v0mod.GRID_CEILING,
    gap=v0mod.GRID_GAP,
    buy_fee=v0mod.BUY_FEE,
    sell_fee=v0mod.SELL_FEE,
)

result = v0mod.run_fixed_grid_backtest(
    df_price=df_1m,
    grid=grid,
    initial_capital=v0mod.INITIAL_CAPITAL,
)
summary = result['summary']
audit_checks = v0mod.audit_v0(result)
AUDIT_STATUS = 'PASS' if all(audit_checks.values()) else 'FAIL'

print('===== V0 CONFIGURATION =====')
print(f'Symbol          : {v0mod.SYMBOL}')
print(f'Period          : {v0mod.START_DATE} -> {v0mod.END_DATE}')
print(f'Initial Capital : {v0mod.INITIAL_CAPITAL:,.2f} USDT')
print(f'Floor           : {v0mod.GRID_FLOOR:,.2f} USDT')
print(f'Ceiling         : {v0mod.GRID_CEILING:,.2f} USDT')
print(f'Gap             : {v0mod.GRID_GAP:,.2f} USDT')
print(f'Number of Grids : {len(grid)}')
print(f'Capital / Grid  : {grid["capital_per_grid"].iloc[0]:,.6f} USDT')

print('\n===== V0 RESULT =====')
print(f'Final Equity    : {summary["final_equity"]:,.2f} USDT')
print(f'Net Return      : {summary["net_return"]:.2%}')
print(f'Annualized      : {summary["annualized_return"]:.2%}')
print(f'Max Drawdown    : {summary["max_drawdown"]:.2%}')
print(f'Calmar Ratio    : {summary["calmar_ratio"]:.3f}')
print(f'Cycles          : {summary["completed_cycles"]:,}')
print(f'Open Positions  : {summary["open_positions"]:,}')
print(f'Final Cash      : {summary["final_cash"]:,.2f} USDT')
print(f'Final BTC       : {summary["final_btc"]:.8f} BTC')

print('\n===== V0 AUDIT =====')
for name, passed in audit_checks.items():
    print(f'{name:32s}: {"PASS" if passed else "FAIL"}')
print(f'Overall                         : {AUDIT_STATUS}')

if AUDIT_STATUS != 'PASS':
    raise AssertionError('V0 AUDIT FAILED')

## 3. Build and upload latest V0 log

In [ ]:
def json_safe(value):
    if isinstance(value, dict):
        return {k: json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return None if not np.isfinite(value) else float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if value is pd.NaT:
        return None
    return value

log_payload = {
    'log_schema_version': 1,
    'strategy': 'V0 Fixed Grid',
    'run_info': {
        'generated_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'repository': REPO,
        'branch': BRANCH,
        'source_file': SOURCE_FILE,
        'notebook': 'Grid_trading_V0.ipynb',
        'symbol': v0mod.SYMBOL,
        'timeframe': v0mod.TIMEFRAME,
        'start_date': v0mod.START_DATE,
        'end_date': v0mod.END_DATE,
        'data_rows': int(len(df_1m)),
        'data_first_time': df_1m['open_time'].min().isoformat(),
        'data_last_time': df_1m['open_time'].max().isoformat(),
    },
    'parameters': {
        'initial_capital': v0mod.INITIAL_CAPITAL,
        'floor': v0mod.GRID_FLOOR,
        'ceiling': v0mod.GRID_CEILING,
        'gap': v0mod.GRID_GAP,
        'buy_fee': v0mod.BUY_FEE,
        'sell_fee': v0mod.SELL_FEE,
    },
    'derived': {
        'number_of_grids': int(len(grid)),
        'capital_per_grid': float(grid['capital_per_grid'].iloc[0]),
    },
    'market_range_diagnostics': {
        'historical_low': float(df_1m['low'].min()),
        'historical_high': float(df_1m['high'].max()),
        'candles_low_below_floor': int((df_1m['low'] < v0mod.GRID_FLOOR).sum()),
        'candles_high_above_ceiling': int((df_1m['high'] > v0mod.GRID_CEILING).sum()),
    },
    'summary': json_safe(summary),
    'audit': {
        'status': AUDIT_STATUS,
        'checks': json_safe(audit_checks),
    },
}

with open(LOCAL_LOG_PATH, 'w', encoding='utf-8') as f:
    json.dump(log_payload, f, indent=2, allow_nan=False)

print(f'Local log: {LOCAL_LOG_PATH}')

try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    github_token = None

if not github_token:
    print("GitHub upload SKIPPED: Colab Secret 'GITHUB_TOKEN' was not found.")
else:
    api_url = f'https://api.github.com/repos/{REPO}/contents/{GITHUB_LOG_PATH}'
    headers = {
        'Authorization': f'Bearer {github_token}',
        'Accept': 'application/vnd.github+json',
        'X-GitHub-Api-Version': '2022-11-28',
    }
    existing = requests.get(api_url, headers=headers, timeout=30)
    body = {
        'message': 'Update latest V0 backtest log',
        'content': base64.b64encode(
            json.dumps(log_payload, indent=2).encode('utf-8')
        ).decode('utf-8'),
        'branch': BRANCH,
    }
    if existing.status_code == 200:
        body['sha'] = existing.json()['sha']

    upload = requests.put(api_url, headers=headers, json=body, timeout=30)
    upload.raise_for_status()
    print('GitHub log upload: SUCCESS')
    print(f'Path             : {GITHUB_LOG_PATH}')
    print(f'Commit SHA       : {upload.json()["commit"]["sha"]}')